<a href="https://colab.research.google.com/github/icarobregon/AI-Engineering-2026/blob/master/sesion-01_llms-setup/src/Sesi%C3%B3n_01_Nivel_03_Anthropic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 11.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

from anthropic import Anthropic
client = Anthropic()

def calculate_api_cost(response):
    # Precios actualizados según documentación oficial (en USD por millón de tokens)
    # Incluye todos los modelos actuales (no deprecated/retired)
    # Fuente: https://platform.claude.com/docs/en/about-claude/pricing#model-pricing
    PRICING = {
        # Latest Models (Current Generation)
        "claude-opus-4-7": {
            "input": 5.00,      # $5 por millón de tokens de entrada
            "output": 25.00     # $25 por millón de tokens de salida
        },
        "claude-sonnet-4-6": {
            "input": 3.00,
            "output": 15.00
        },
        "claude-haiku-4-5-20251001": {
            "input": 1.00,
            "output": 5.00
        },
        "claude-haiku-4-5": {  # Alias para claude-haiku-4-5-20251001
            "input": 1.00,
            "output": 5.00
        },

        # Legacy Models (Still Available, Not Deprecated)
        "claude-opus-4-6": {
            "input": 5.00,
            "output": 25.00
        },
        "claude-opus-4-5-20251101": {
            "input": 5.00,
            "output": 25.00
        },
        "claude-opus-4-5": {  # Alias para claude-opus-4-5-20251101
            "input": 5.00,
            "output": 25.00
        },
        "claude-sonnet-4-5-20250929": {
            "input": 3.00,
            "output": 15.00
        },
        "claude-sonnet-4-5": {  # Alias para claude-sonnet-4-5-20250929
            "input": 3.00,
            "output": 15.00
        },
        "claude-opus-4-1-20250805": {
            "input": 15.00,
            "output": 75.00
        },
        "claude-opus-4-1": {  # Alias para claude-opus-4-1-20250805
            "input": 15.00,
            "output": 75.00
        },
    }

    model = response.model
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens

    # Validar que el modelo existe en nuestro diccionario
    if model not in PRICING:
        available_models = ", ".join(sorted(set(PRICING.keys())))
        raise ValueError(
            f"❌ Modelo '{model}' no encontrado en tabla de precios.\n"
            f"Modelos disponibles:\n{available_models}\n\n"
            f"ℹ️  Nota: Los modelos deprecated o retired no están soportados.\n"
            f"Para más info: https://platform.claude.com/docs/en/about-claude/model-deprecations"
        )

    # Obtener precios del modelo
    prices = PRICING[model]

    # Calcular costes (precios están en USD por 1M tokens)
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    total_cost = input_cost + output_cost

    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
        "total_cost_formatted": f"${total_cost:.6f}"
    }

def print_cost_analysis(cost_info):
    print(f"\n💰 Análisis de Costes:")
    print(f"   Modelo:         {cost_info['model']}")
    print(f"   Input tokens:   {cost_info['input_tokens']:,}")
    print(f"   Output tokens:  {cost_info['output_tokens']:,}")
    print(f"   Total tokens:   {cost_info['input_tokens'] + cost_info['output_tokens']:,}")
    print(f"   ─────────────────────────────")
    print(f"   Coste input:    ${cost_info['input_cost']:.8f}")
    print(f"   Coste output:   ${cost_info['output_cost']:.8f}")
    print(f"   💵 Total:        {cost_info['total_cost_formatted']}")

def main():
    message = client.messages.create(
      max_tokens=1024,
      system="Actua como un Devops Senior de una empresa de desarrollo de software. Responde de una manera técnica y directa.",
      messages=[
          {
              "role": "user",
              "content": "Quiero alojar un proyecto personal desarrollado en Next.js y que necesita una base de datos PostgreSQL. Recomiéndame como hacerlo.",
          }
      ],
      model="claude-haiku-4-5",
    )

    cost_info = calculate_api_cost(message)
    print_cost_analysis(cost_info)

if __name__ == "__main__":
    main()



💰 Análisis de Costes:
   Modelo:         claude-haiku-4-5-20251001
   Input tokens:   73
   Output tokens:  585
   Total tokens:   658
   ─────────────────────────────
   Coste input:    $0.00007300
   Coste output:   $0.00292500
   💵 Total:        $0.002998
